# ML-Based Cloud Resource Intelligence and Workload Optimization

**Module 1 — bounded data loading.** This notebook uses a small subset of the [Google Cluster Data 2011-2 trace](https://github.com/google/cluster-data/blob/master/ClusterData2011_2.md), released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Run cells from top to bottom in a Google Colab CPU runtime. The full compressed trace is about 41 GB; this notebook downloads only a bounded selection of task-usage shards. CPU and memory values are normalized trace measurements, not directly cores or GB.

The first run needs internet and a Google Drive mount. Set `PROJECT_DIR` in the configuration cell to another Drive folder if desired. Cached files are reused on later runs. No Google Cloud login or `gcloud` command is needed. The `data/` directory stays inside the configured project folder; ensure that Drive has 2.5 GB free before running.


In [8]:
# Colab has pandas and requests. Install the Parquet engine only if necessary.
import importlib.util
if importlib.util.find_spec("pyarrow") is None:
    %pip -q install pyarrow

from pathlib import Path
import base64
import hashlib
import json
import math
import os
import re
import time
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pyarrow as pa
import pyarrow.parquet as pq

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Change this one path if your Drive project folder has another name.
PROJECT_DIR = Path("/content/drive/MyDrive/ml-project") if IN_COLAB else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
for directory in (RAW_DIR, PROCESSED_DIR, ARTIFACTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
PILOT_NAME = "task_usage/part-00000-of-00500.csv.gz"
MAX_SHARDS = 6                    # Pilot covered only 85 minutes; six adjacent shards target a longer window.
MAX_COMPRESSED_BYTES = 1_000_000_000
MAX_PROJECT_BYTES = 2_500_000_000 # data/ + artifacts/; does not include this notebook.
MAX_ROWS = 300_000
CHUNK_ROWS = 50_000
INITIAL_TASK_FRACTION = 0.10
SCHEMA_URL = "https://storage.googleapis.com/clusterdata-2011-2/schema.csv"
BUCKET_API = "https://storage.googleapis.com/storage/v1/b/clusterdata-2011-2/o"
OBJECT_BASE = "https://storage.googleapis.com/clusterdata-2011-2/"
MANIFEST_PATH = ARTIFACTS_DIR / "module1_manifest.json"
PARQUET_PATH = PROCESSED_DIR / "task_usage_subset.parquet"

assert 1 <= MAX_SHARDS <= 500
assert 0 < INITIAL_TASK_FRACTION <= 1
assert MAX_PROJECT_BYTES <= 2_500_000_000 and MAX_COMPRESSED_BYTES <= 1_000_000_000
print("Project:", PROJECT_DIR)
print("Limits: compressed <=", MAX_COMPRESSED_BYTES, "bytes; data/artifacts <=", MAX_PROJECT_BYTES, "bytes; rows <=", MAX_ROWS)


Project: /content/drive/MyDrive/ml-project
Limits: compressed <= 1000000000 bytes; data/artifacts <= 2500000000 bytes; rows <= 300000


## Official schema and shard manifest

The schema supplies the column names and positions. Object metadata supplies actual compressed sizes and checksums; the notebook never assumes all shards are the same size. The pilot covered only 85 minutes, so the default now selects six adjacent shards. Increase or lower `MAX_SHARDS` after inspecting the new coverage. Shards are selected in numeric order, but coverage is checked from the rows rather than inferred from shard numbers.

**Pilot result from the first Colab run:** shard 00000 was 91,723,415 compressed bytes, had 2,520,773 rows, and covered trace time 600?5,700 seconds (85 minutes). A 10% identity sample retained 254,404 rows and 234,465 contiguous pairs. The pilot proved the loader works, but a wider time range will make later temporal splits more useful.


In [10]:
def http_session():
    session = requests.Session()
    retries = Retry(total=3, connect=3, read=3, backoff_factor=1,
                    status_forcelist=(429, 500, 502, 503, 504))
    session.mount("https://", HTTPAdapter(max_retries=retries))
    return session

HTTP = http_session()

def project_bytes():
    return sum(p.stat().st_size for folder in (PROJECT_DIR / "data", ARTIFACTS_DIR)
               for p in folder.rglob("*") if p.is_file())

def ensure_space(additional=0):
    current = project_bytes()
    if current + additional > MAX_PROJECT_BYTES:
        raise RuntimeError(f"Project data/artifacts would exceed {MAX_PROJECT_BYTES:,} bytes: "
                           f"{current:,} existing + {additional:,} proposed")

def atomic_json(path, obj):
    encoded = json.dumps(obj, indent=2, ensure_ascii=False).encode("utf-8")
    ensure_space(2 * len(encoded))  # Temporary file plus final file.
    temp = path.with_suffix(path.suffix + ".part")
    temp.write_bytes(encoded)
    os.replace(temp, path)

def download_public(url, path, expected_size=None, md5=None):
    """Download once, verifying cached files too. Failed partials are removed."""
    def verified(p):
        if not p.exists(): return False
        if expected_size is not None and p.stat().st_size != int(expected_size): return False
        if md5:
            digest = hashlib.md5()  # Used only to match GCS's published checksum.
            with p.open("rb") as f:
                for block in iter(lambda: f.read(1024 * 1024), b""):
                    digest.update(block)
            return base64.b64encode(digest.digest()).decode() == md5
        return True

    if verified(path): return "cached"
    if path.exists():
        raise RuntimeError(f"Existing file failed verification: {path}. Inspect it before rerunning.")
    ensure_space(int(expected_size) if expected_size is not None else 100_000_000)
    temp = path.with_suffix(path.suffix + ".part")
    if temp.exists(): temp.unlink()  # A previous interrupted transfer cannot be trusted.
    try:
        with HTTP.get(url, stream=True, timeout=(15, 90)) as response:
            response.raise_for_status()
            content_length = response.headers.get("Content-Length")
            if expected_size is not None and content_length and int(content_length) != int(expected_size):
                raise RuntimeError("Server size differs from object metadata")
            written = 0
            with temp.open("wb") as f:
                for block in response.iter_content(chunk_size=1024 * 1024):
                    if not block: continue
                    ensure_space(len(block))
                    f.write(block)
                    written += len(block)
                    if expected_size is not None and written > int(expected_size):
                        raise RuntimeError("Download exceeded published size")
        if not verified(temp): raise RuntimeError(f"Size/checksum mismatch for {path.name}")
        os.replace(temp, path)
    except Exception:
        temp.unlink(missing_ok=True)
        raise
    return "downloaded"

SCHEMA_PATH = RAW_DIR / "schema.csv"
schema_status = download_public(SCHEMA_URL, SCHEMA_PATH)
schema = pd.read_csv(SCHEMA_PATH)
required_schema_fields = {"file pattern", "field number", "content"}
if not required_schema_fields.issubset(schema.columns):
    raise ValueError(f"Unexpected schema columns: {schema.columns.tolist()}")
usage_schema = schema[schema["file pattern"].astype(str).str.startswith("task_usage/")].copy()
usage_schema["field number"] = pd.to_numeric(usage_schema["field number"], errors="raise")
usage_schema = usage_schema.sort_values("field number")
if usage_schema["field number"].tolist() != list(range(1, 21)):
    raise ValueError("Expected exactly 20 consecutive task_usage fields in official schema")
COLUMNS = usage_schema["content"].astype(str).str.strip().tolist()
if len(set(COLUMNS)) != 20 or COLUMNS[:5] != ["start time", "end time", "job ID", "task index", "machine ID"]:
    raise ValueError("Official task_usage schema layout changed; inspect before continuing")
print("Schema:", schema_status)
display(usage_schema[["field number", "content", "format", "mandatory"]].reset_index(drop=True))


Schema: cached


,field number,content,format,mandatory
0,1,start time,INTEGER,YES
1,2,end time,INTEGER,YES
2,3,job ID,INTEGER,YES
3,4,task index,INTEGER,YES
4,5,machine ID,INTEGER,YES
5,6,CPU rate,FLOAT,NO
6,7,canonical memory usage,FLOAT,NO
7,8,assigned memory usage,FLOAT,NO
8,9,unmapped page cache,FLOAT,NO
9,10,total page cache,FLOAT,NO


In [11]:
def list_task_usage_objects():
    objects = []
    token = None
    while True:
        params = {"prefix": "task_usage/", "maxResults": 1000,
                  "fields": "nextPageToken,items(name,size,md5Hash,crc32c)"}
        if token: params["pageToken"] = token
        response = HTTP.get(BUCKET_API, params=params, timeout=(15, 90))
        response.raise_for_status()
        payload = response.json()
        objects.extend(payload.get("items", []))
        token = payload.get("nextPageToken")
        if not token: break
    objects = [x for x in objects if re.fullmatch(r"task_usage/part-\d{5}-of-00500\.csv\.gz", x["name"])]
    objects.sort(key=lambda x: x["name"])
    if len(objects) != 500 or objects[0]["name"] != PILOT_NAME:
        raise ValueError(f"Unexpected task_usage listing: {len(objects)} matching objects")
    return objects

objects = list_task_usage_objects()
selected = objects[:MAX_SHARDS]
compressed_bytes = sum(int(x["size"]) for x in selected)
if compressed_bytes > MAX_COMPRESSED_BYTES:
    raise RuntimeError(f"Selection needs {compressed_bytes:,} compressed bytes; reduce MAX_SHARDS")
ensure_space(sum(int(x["size"]) for x in selected if not (RAW_DIR / Path(x["name"]).name).exists()))
manifest = {"source": "Google Cluster Data 2011-2", "schema_url": SCHEMA_URL,
            "seed": SEED, "selected": [], "compressed_budget": MAX_COMPRESSED_BYTES,
            "project_budget": MAX_PROJECT_BYTES}
for obj in selected:
    name = obj["name"]
    path = RAW_DIR / Path(name).name
    status = download_public(OBJECT_BASE + quote(name, safe="/"), path,
                             expected_size=int(obj["size"]), md5=obj.get("md5Hash"))
    manifest["selected"].append({"name": name, "size": int(obj["size"]),
                                 "md5Hash": obj.get("md5Hash"), "crc32c": obj.get("crc32c"),
                                 "status": status, "local_path": str(path)})
    print(name, f'{int(obj["size"])/1e6:.1f} MB', status)
atomic_json(MANIFEST_PATH, manifest)
print("Selected compressed bytes:", f"{compressed_bytes:,}", "| Project data/artifacts:", f"{project_bytes():,}")


task_usage/part-00000-of-00500.csv.gz 91.7 MB cached
task_usage/part-00001-of-00500.csv.gz 87.2 MB downloaded
task_usage/part-00002-of-00500.csv.gz 90.9 MB downloaded
task_usage/part-00003-of-00500.csv.gz 85.9 MB downloaded
task_usage/part-00004-of-00500.csv.gz 91.9 MB downloaded
task_usage/part-00005-of-00500.csv.gz 94.5 MB downloaded
Selected compressed bytes: 542,159,876 | Project data/artifacts: 549,402,294


## Bounded, reproducible task sample

We scan the selected gzip files in 50,000-row chunks and retain task identities, so their time histories stay together. A fixed integer mixing function selects the same `(job ID, task index)` across chunks and runs. If the sample would exceed 300,000 rows, the fraction is reduced and the cached gzip files are rescanned. The source gzip files are never expanded to CSV. The final file is written incrementally as Parquet.


In [12]:
# Positions come from the verified official schema above.
KEEP_FIELDS = ["start time", "end time", "job ID", "task index", "machine ID",
               "CPU rate", "canonical memory usage", "assigned memory usage",
               "maximum memory usage", "disk I/O time", "local disk space usage",
               "maximum CPU rate", "maximum disk IO time", "sampled CPU usage"]
KEEP_POS = [COLUMNS.index(name) for name in KEEP_FIELDS]
DTYPES = {i: ("Int64" if COLUMNS[i] == "machine ID" else "int64")
          for i in KEEP_POS if COLUMNS[i] in
          ("start time", "end time", "job ID", "task index", "machine ID")}
DTYPES.update({i: "float32" for i in KEEP_POS if i not in DTYPES})

def iter_usage():
    for entry in manifest["selected"]:
        path = Path(entry["local_path"])
        for chunk in pd.read_csv(path, compression="gzip", header=None, usecols=KEEP_POS,
                                 dtype=DTYPES, chunksize=CHUNK_ROWS):
            chunk.columns = [COLUMNS[i] for i in KEEP_POS]
            yield entry["name"], chunk

def task_mask(frame, fraction):
    # SplitMix64-style deterministic integer mixing. No Python hash() or row sampling.
    job = frame["job ID"].to_numpy(dtype=np.uint64, copy=False)
    task = frame["task index"].to_numpy(dtype=np.uint64, copy=False)
    with np.errstate(over="ignore"):
        z = job ^ (task * np.uint64(0x9E3779B97F4A7C15)) ^ np.uint64(SEED)
        z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        z = z ^ (z >> np.uint64(31))
    if fraction >= 1: return np.ones(len(frame), dtype=bool)
    return z < np.uint64(int(fraction * (2**64 - 1)))

def scan_fraction(fraction):
    selected_rows = 0
    all_rows = 0
    min_start, max_end = None, None
    by_shard = {}
    for name, chunk in iter_usage():
        all_rows += len(chunk)
        starts = chunk["start time"]
        ends = chunk["end time"]
        min_start = int(starts.min()) if min_start is None else min(min_start, int(starts.min()))
        max_end = int(ends.max()) if max_end is None else max(max_end, int(ends.max()))
        kept = int(task_mask(chunk, fraction).sum())
        selected_rows += kept
        stats = by_shard.setdefault(name, {"rows": 0, "selected_rows": 0,
                                           "min_start": None, "max_end": None})
        stats["rows"] += len(chunk)
        stats["selected_rows"] += kept
        stats["min_start"] = int(starts.min()) if stats["min_start"] is None else min(stats["min_start"], int(starts.min()))
        stats["max_end"] = int(ends.max()) if stats["max_end"] is None else max(stats["max_end"], int(ends.max()))
    return {"all_rows": all_rows, "selected_rows": selected_rows,
            "min_start": min_start, "max_end": max_end, "by_shard": by_shard}

fraction = INITIAL_TASK_FRACTION
for attempt in range(12):
    scan = scan_fraction(fraction)
    print(f"Pass {attempt+1}: fraction={fraction:.7f}; retained={scan['selected_rows']:,}/{scan['all_rows']:,}")
    if 0 < scan["selected_rows"] <= MAX_ROWS: break
    fraction /= 2
else:
    raise RuntimeError("Could not choose a nonempty task sample under the row limit")
manifest["sample_fraction"] = fraction
manifest["source_rows"] = scan["all_rows"]
manifest["selected_rows"] = scan["selected_rows"]
manifest["source_time_range_us"] = [scan["min_start"], scan["max_end"]]
manifest["shard_coverage"] = scan["by_shard"]
coverage = pd.DataFrame.from_dict(scan["by_shard"], orient="index")
coverage["hours_covered"] = (coverage["max_end"] - coverage["min_start"]) / 3_600_000_000
print("Source time coverage by shard (hours; overlaps are possible):")
display(coverage)
atomic_json(MANIFEST_PATH, manifest)


Pass 1: fraction=0.1000000; retained=1,481,237/14,779,583
Pass 2: fraction=0.0500000; retained=737,177/14,779,583
Pass 3: fraction=0.0250000; retained=370,281/14,779,583
Pass 4: fraction=0.0125000; retained=184,426/14,779,583
Source time coverage by shard (hours; overlaps are possible):


,rows,selected_rows,min_start,max_end,hours_covered
task_usage/part-00000-of-00500.csv.gz,2520773,31907,600000000,5700000000,1.416667
task_usage/part-00001-of-00500.csv.gz,2352447,29422,5612000000,10800000000,1.441111
task_usage/part-00002-of-00500.csv.gz,2482505,30670,10623000000,15900000000,1.465833
task_usage/part-00003-of-00500.csv.gz,2346031,29120,15634000000,20700000000,1.407222
task_usage/part-00004-of-00500.csv.gz,2508478,31399,20645000000,25800000000,1.431944
task_usage/part-00005-of-00500.csv.gz,2569349,31908,25656000000,30900000000,1.456667


In [13]:
# Write once, incrementally. The .part file is removed after an error.
TEMP_PARQUET = PARQUET_PATH.with_suffix(".parquet.part")
if TEMP_PARQUET.exists(): TEMP_PARQUET.unlink()
writer = None
written = 0
try:
    try:
        for _, chunk in iter_usage():
            small = chunk.loc[task_mask(chunk, fraction)]
            if small.empty: continue
            written += len(small)
            if written > MAX_ROWS: raise RuntimeError("Row cap exceeded during Parquet write")
            table = pa.Table.from_pandas(small, preserve_index=False)
            if writer is None: writer = pq.ParquetWriter(TEMP_PARQUET, table.schema, compression="zstd")
            writer.write_table(table)
            # Existing final Parquet is counted until replaced; temp also counts.
            ensure_space()
    finally:
        if writer is not None: writer.close()
    if written == 0: raise RuntimeError("Sample has no rows")
    ensure_space()
    os.replace(TEMP_PARQUET, PARQUET_PATH)
except Exception:
    TEMP_PARQUET.unlink(missing_ok=True)
    raise

subset = pd.read_parquet(PARQUET_PATH)
assert len(subset) == written == scan["selected_rows"] <= MAX_ROWS
subset.sort_values(["job ID", "task index", "start time"], inplace=True, kind="stable")
subset_memory = int(subset.memory_usage(deep=True).sum())
task_count = int(subset[["job ID", "task index"]].drop_duplicates().shape[0])
next_start = subset.groupby(["job ID", "task index"], sort=False)["start time"].shift(-1)
contiguous_pairs = int(next_start.eq(subset["end time"]).sum())
summary = {"source_rows": scan["all_rows"], "retained_rows": written,
           "retained_tasks": task_count, "contiguous_next_intervals": contiguous_pairs,
           "source_time_range_us": [scan["min_start"], scan["max_end"]],
           "source_hours_covered": round((scan["max_end"] - scan["min_start"]) / 3_600_000_000, 2),
           "subset_time_range_us": [int(subset["start time"].min()), int(subset["end time"].max())],
           "compressed_bytes": compressed_bytes, "parquet_bytes": PARQUET_PATH.stat().st_size,
           "dataframe_memory_bytes": subset_memory, "project_data_artifact_bytes": project_bytes()}
manifest["summary"] = summary
atomic_json(MANIFEST_PATH, manifest)
print(json.dumps(summary, indent=2))
print("Subset preview:")
display(subset[KEEP_FIELDS].head(5))
print("Review the per-shard coverage above for gaps/overlap before using the subset for forecasting.")
print("This is an identity-hash sample from a limited time slice, not a random sample of all 29 days.")
print("A low contiguous-pair count means Module 2 needs a wider selection or a different sampling fraction.")


{
  "source_rows": 14779583,
  "retained_rows": 184426,
  "retained_tasks": 4124,
  "contiguous_next_intervals": 178427,
  "source_time_range_us": [
    600000000,
    30900000000
  ],
  "source_hours_covered": 8.42,
  "subset_time_range_us": [
    600000000,
    30900000000
  ],
  "compressed_bytes": 542159876,
  "parquet_bytes": 7867467,
  "dataframe_memory_bytes": 15676210,
  "project_data_artifact_bytes": 550034426
}
Subset preview:


,start time,end time,job ID,task index,machine ID,CPU rate,canonical memory usage,assigned memory usage,maximum memory usage,disk I/O time,local disk space usage,maximum CPU rate,maximum disk IO time,sampled CPU usage
0,600000000,900000000,3418329,2,336025676,0.002533,0.09094,0.1008,0.09106,0.000003,0.000358,0.029720,0.000357,0.0
1829,900000000,1200000000,3418329,2,336025676,0.002663,0.09094,0.1008,0.09106,0.000007,0.000358,0.027680,0.001358,0.0
3640,1200000000,1500000000,3418329,2,336025676,0.002316,0.09094,0.1008,0.09106,0.000003,0.000358,0.023500,0.000143,0.0
5641,1500000000,1800000000,3418329,2,336025676,0.002205,0.09094,0.1008,0.09106,0.000003,0.000358,0.015240,0.000143,0.0
7609,1800000000,2100000000,3418329,2,336025676,0.002014,0.09094,0.1008,0.09106,0.000003,0.000358,0.009232,0.000143,0.0


Review the per-shard coverage above for gaps/overlap before using the subset for forecasting.
This is an identity-hash sample from a limited time slice, not a random sample of all 29 days.
A low contiguous-pair count means Module 2 needs a wider selection or a different sampling fraction.


## Module 1 complete ? preparation notebook

The six-shard Colab run covered 8.42 hours, retained 184,426 rows, and found 178,427 contiguous next-interval pairs. The saved Parquet file is about 7.9 MB; `analysis.ipynb` loads it directly for Modules 2?5. You do not need to rerun this notebook for normal modeling work. Rerun it only if you change the shard selection, sampling settings, or preprocessing; cached gzip files will be reused, but the scan and Parquet write will repeat.
